In [1]:
import os
import numpy as np
import mne
from pathlib import Path

In [2]:
subjects_dir = mne.datasets.fetch_fsaverage(verbose=True)
subjects_dir = Path(subjects_dir).parent
subjects_dir
subject = 'fsaverage'

0 files missing from root.txt in C:\Users\hugop\mne_data\MNE-fsaverage-data
0 files missing from bem.txt in C:\Users\hugop\mne_data\MNE-fsaverage-data\fsaverage


In [3]:
src = mne.setup_source_space(subject = subject, spacing='ico5', subjects_dir=subjects_dir, add_dist=False)

Setting up the source space with the following parameters:

SUBJECTS_DIR = C:\Users\hugop\mne_data\MNE-fsaverage-data
Subject      = fsaverage
Surface      = white
Icosahedron subdivision grade 5

>>> 1. Creating the source space...

Doing the icosahedral vertex picking...
Loading C:\Users\hugop\mne_data\MNE-fsaverage-data\fsaverage\surf\lh.white...
Mapping lh fsaverage -> ico (5) ...
    Triangle neighbors and vertex normals...
Loading geometry from C:\Users\hugop\mne_data\MNE-fsaverage-data\fsaverage\surf\lh.sphere...
Setting up the triangulation for the decimated surface...
loaded lh.white 10242/163842 selected to source space (ico = 5)

Loading C:\Users\hugop\mne_data\MNE-fsaverage-data\fsaverage\surf\rh.white...
Mapping rh fsaverage -> ico (5) ...
    Triangle neighbors and vertex normals...
Loading geometry from C:\Users\hugop\mne_data\MNE-fsaverage-data\fsaverage\surf\rh.sphere...
Setting up the triangulation for the decimated surface...
loaded rh.white 10242/163842 selected to 

In [94]:
hemi_l = 0

rr_l = src[hemi_l]['rr']
vertno_l = src[hemi_l]['vertno'] 
coords_l = rr_l[vertno_l] 


hemi_r = 1

rr_r = src[hemi_r]['rr']
vertno_r = src[hemi_r]['vertno'] 
coords_r = rr_r[vertno_r] 

In [141]:
def make_mask(coords, px = (0,1),py= (0,1),pz= (0,1), reverse = False):

    def calcul(min,max,p):
        return min + p * (max - min)
    
    if reverse :
        px = (1 - px[1],1 - px[0])

    minx,miny,minz = coords.min(axis=0)
    maxx, maxy, maxz = coords.max(axis=0)

    x = coords[:, 0]
    y = coords[:, 1]
    z = coords[:, 2]

    x_inf = calcul(minx,maxx,px[0])
    x_sup = calcul(minx,maxx,px[1])

    y_inf = calcul(miny,maxy,py[0])
    y_sup = calcul(miny,maxy,py[1])

    z_inf = calcul(minz,maxz,pz[0])
    z_sup = calcul(minz,maxz,pz[1])


    mask = ((x <= x_sup) & (x >= x_inf) & 
             (y <= y_sup) & (y >= y_inf) & 
             (z <= z_sup) & (z >= z_inf))

    return mask

def fusion_mask_list(liste):
    
    def fusion_mask(mask_1, mask_2) :
        mask_out = []
        if len(mask_1) == 0 or mask_1 is None:
            return mask_2
        else :
            for m1, m2 in zip(mask_1,mask_2):
                if m1 == False and m2 == False:
                    mask_out.append(False)
                else :
                    mask_out.append(True)
        return mask_out
    
    mask_out = []
    for mask in liste:
        mask_out = fusion_mask(mask_out, mask)
        
    return np.array(mask_out)


In [142]:
brain = mne.viz.Brain('fsaverage', hemi='both', surf='inflated')
###################### coordonee temporal ############################
# gauche

mask_temp_2_l = make_mask(coords_l,py=(0.35,0.47),pz=(0,0.5),px = (0,0.37))
mask_temp_1_bis_l = make_mask(coords_l,py=(0.47,0.6),pz=(0,0.42),px = (0,0.36))
mask_temp_1_l = make_mask(coords_l,py=(0.47,0.75),pz=(0,0.366),px = (0,0.36))
mask_temp_3_l = make_mask(coords_l,py=(0.22,0.35),pz=(0,0.55),px = (0,0.45))

# droit
mask_temp_2_r = make_mask(coords_r,py=(0.35,0.47),pz=(0,0.5),px = (0,0.37),reverse = True)
mask_temp_1_bis_r = make_mask(coords_r,py=(0.47,0.6),pz=(0,0.42),px = (0,0.36),reverse = True)
mask_temp_1_r = make_mask(coords_r,py=(0.47,0.75),pz=(0,0.366),px = (0,0.36),reverse = True)
mask_temp_3_r = make_mask(coords_r,py=(0.22,0.35),pz=(0,0.55),px = (0,0.45),reverse = True)

###################### coordonee occipital ############################
# gauche
mask_occ_1_l = make_mask(coords_l,py=(0,0.13),pz=(0,0.7),px = (0.58,1))
mask_occ_2_l = make_mask(coords_l,py=(0,0.22),pz=(0,0.7),px = (0,0.58))
mask_occ_3_l = make_mask(coords_l,py=(0,0.18),pz=(0.65,0.8),px = (0,0.9))
mask_occ_4_l = make_mask(coords_l,py=(0,0.2),pz=(0.45,1),px = (0,0.65))
mask_occ_5_l = make_mask(coords_l,py=(0.2,0.26),pz=(0.58,0.715),px = (0.55,0.69))

# droit
mask_occ_1_r = make_mask(coords_r,py=(0,0.13),pz=(0,0.7),px = (0.58,1),reverse = True)
mask_occ_2_r = make_mask(coords_r,py=(0,0.22),pz=(0,0.7),px = (0,0.58),reverse = True)
mask_occ_3_r = make_mask(coords_r,py=(0,0.18),pz=(0.65,0.8),px = (0,0.9),reverse = True)
mask_occ_4_r = make_mask(coords_r,py=(0,0.2),pz=(0.45,1),px = (0,0.65),reverse = True)
mask_occ_5_r = make_mask(coords_r,py=(0.2,0.26),pz=(0.58,0.715),px = (0.55,0.69),reverse = True)
###################### coordonee TPJ - Parietal ############################
# gauche
mask_TPJ_parietal1_l = make_mask(coords_l,py=(0,0.22),pz=(0.7,1),px = (0,0.7))
mask_TPJ_parietal2_l = make_mask(coords_l,py=(0.22,0.42),pz=(0.55,1),px = (0,0.7))
mask_TPJ_parietal3_l = make_mask(coords_l,py=(0,0.22),pz=(0.7,1),px = (0,0.87))
mask_TPJ_parietal4_l = make_mask(coords_l,py=(0.22,0.32),pz=(0.83,1),px = (0,0.85))
mask_TPJ_parietal5_l = make_mask(coords_l,py=(0.32,0.43),pz=(0.89,1),px = (0,0.95))

# droit
mask_TPJ_parietal1_r = make_mask(coords_r,py=(0,0.22),pz=(0.7,1),px = (0,0.7),reverse = True)
mask_TPJ_parietal2_r = make_mask(coords_r,py=(0.22,0.42),pz=(0.55,1),px = (0,0.7),reverse = True)
mask_TPJ_parietal3_r = make_mask(coords_r,py=(0,0.22),pz=(0.7,1),px = (0,0.87),reverse = True)
mask_TPJ_parietal4_r = make_mask(coords_r,py=(0.22,0.32),pz=(0.83,1),px = (0,0.85),reverse = True)
mask_TPJ_parietal5_r = make_mask(coords_r,py=(0.32,0.43),pz=(0.89,1),px = (0,0.95),reverse = True)


###################### coordonee Pericentral ############################
# gauche
mask_pericentral_1_l = make_mask(coords_l,py=(0.42, 0.58),pz=(0.5,1),px = (0,0.7))
mask_pericentral_2_l = make_mask(coords_l,py=(0.42, 0.58),pz=(0.85,1),px = (0,0.9))

# droit
mask_pericentral_1_r = make_mask(coords_r,py=(0.42, 0.58),pz=(0.5,1),px = (0,0.7),reverse = True)
mask_pericentral_2_r = make_mask(coords_r,py=(0.42, 0.58),pz=(0.85,1),px = (0,0.9),reverse = True)
############################## coordonee IFG #################################
# gauche
mask_ifg_1_l = make_mask(coords_l,py=(0.58, 0.75),pz=(0.45,0.7),px = (0,0.5))

# droit
mask_ifg_1_r = make_mask(coords_r,py=(0.58, 0.75),pz=(0.45,0.7),px = (0,0.5),reverse = True)


###################### coordonee globales ############################

mask_TPJ_parietal_l = fusion_mask_list([mask_TPJ_parietal1_l,mask_TPJ_parietal2_l,mask_TPJ_parietal3_l,mask_TPJ_parietal4_l])
mask_temp_l = fusion_mask_list([mask_temp_1_l,mask_temp_1_bis_l,mask_temp_2_l,mask_temp_3_l])
mask_occ_l = fusion_mask_list([mask_occ_1_l,mask_occ_2_l,mask_occ_3_l,mask_occ_4_l,mask_occ_5_l])
mask_pericentral_l = fusion_mask_list([mask_pericentral_1_l,mask_pericentral_2_l])

mask_TPJ_parietal_r = fusion_mask_list([mask_TPJ_parietal1_r,mask_TPJ_parietal2_r,mask_TPJ_parietal3_r,mask_TPJ_parietal4_r])
mask_temp_r = fusion_mask_list([mask_temp_1_r,mask_temp_1_bis_r,mask_temp_2_r,mask_temp_3_r])
mask_occ_r = fusion_mask_list([mask_occ_1_r,mask_occ_2_r,mask_occ_3_r,mask_occ_4_r,mask_occ_5_r])
mask_pericentral_r = fusion_mask_list([mask_pericentral_1_r,mask_pericentral_2_r])



temp_1_l = mne.Label(vertno_l[mask_temp_1_l], hemi='lh', name='test')
temp_1_bis_l = mne.Label(vertno_l[mask_temp_1_bis_l], hemi='lh', name='test')
temp_2_l = mne.Label(vertno_l[mask_temp_2_l], hemi='lh', name='test')
temp_3_l = mne.Label(vertno_l[mask_temp_3_l], hemi='lh', name='test')

temp_1 = mne.Label(vertno_r[mask_temp_1_r], hemi='rh', name='test')
temp_1_bis = mne.Label(vertno_r[mask_temp_1_bis_r], hemi='rh', name='test')
temp_2 = mne.Label(vertno_r[mask_temp_2_r], hemi='rh', name='test')
temp_3 = mne.Label(vertno_r[mask_temp_3_r], hemi='rh', name='test')



tpj_1_l = mne.Label(vertno_l[mask_TPJ_parietal1_l], hemi='lh', name='test2')
tpj_2_l = mne.Label(vertno_l[mask_TPJ_parietal2_l], hemi='lh', name='test2')
tpj_3_l = mne.Label(vertno_l[mask_TPJ_parietal3_l], hemi='lh', name='test2')
tpj_4_l = mne.Label(vertno_l[mask_TPJ_parietal4_l], hemi='lh', name='test2')
tpj_5_l = mne.Label(vertno_l[mask_TPJ_parietal5_l], hemi='lh', name='test2')

tpj_1_r = mne.Label(vertno_r[mask_TPJ_parietal1_r], hemi='rh', name='test2')
tpj_2_r = mne.Label(vertno_r[mask_TPJ_parietal2_r], hemi='rh', name='test2')
tpj_3_r = mne.Label(vertno_r[mask_TPJ_parietal3_r], hemi='rh', name='test2')
tpj_4_r = mne.Label(vertno_r[mask_TPJ_parietal4_r], hemi='rh', name='test2')
tpj_5_r = mne.Label(vertno_r[mask_TPJ_parietal5_r], hemi='rh', name='test2')



occ_1_l = mne.Label(vertno_l[mask_occ_1_l], hemi='lh', name='test2')
occ_2_l = mne.Label(vertno_l[mask_occ_2_l], hemi='lh', name='test2')
occ_3_l = mne.Label(vertno_l[mask_occ_3_l], hemi='lh', name='test2')
occ_4_l = mne.Label(vertno_l[mask_occ_4_l], hemi='lh', name='test2')
occ_5_l = mne.Label(vertno_l[mask_occ_5_l], hemi='lh', name='test2')

occ_1_r = mne.Label(vertno_r[mask_occ_1_r], hemi='rh', name='test2')
occ_2_r = mne.Label(vertno_r[mask_occ_2_r], hemi='rh', name='test2')
occ_3_r = mne.Label(vertno_r[mask_occ_3_r], hemi='rh', name='test2')
occ_4_r = mne.Label(vertno_r[mask_occ_4_r], hemi='rh', name='test2')
occ_5_r = mne.Label(vertno_r[mask_occ_5_r], hemi='rh', name='test2')



pericentral_1_l = mne.Label(vertno_l[mask_pericentral_1_l], hemi='lh', name='test2')
pericentral_2_l = mne.Label(vertno_l[mask_pericentral_2_l], hemi='lh', name='test2')

pericentral_1_r = mne.Label(vertno_r[mask_pericentral_1_r], hemi='rh', name='test2')
pericentral_2_r = mne.Label(vertno_r[mask_pericentral_2_r], hemi='rh', name='test2')



ifg_1_l = mne.Label(vertno_l[mask_ifg_1_l], hemi='lh', name='test2')

ifg_1_r = mne.Label(vertno_r[mask_ifg_1_r], hemi='rh', name='test2')

################ affichage sous temporal ###################

# brain.add_label(temp_1_l, color='yellow')
# brain.add_label(temp_1_bis_l, color='green')
# brain.add_label(temp_2_l, color='red')
# brain.add_label(temp_3_l, color='blue')

# brain.add_label(temp_1_r, color='yellow')
# brain.add_label(temp_1_bis_r, color='green')
# brain.add_label(temp_2_r, color='red')
# brain.add_label(temp_3_r, color='blue')

################ affichage sous TPJ ###################

# brain.add_label(tpj_1_l, color='yellow')
# brain.add_label(tpj_2_l, color='green')
# brain.add_label(tpj_3_l, color='yellow')
# brain.add_label(tpj_4_l, color='blue')
# brain.add_label(tpj_5_l, color='purple')

# brain.add_label(tpj_1_r, color='yellow')
# brain.add_label(tpj_2_r, color='green')
# brain.add_label(tpj_3_r, color='yellow')
# brain.add_label(tpj_4_r, color='blue')
# brain.add_label(tpj_5_r, color='purple')


################ affichage sous occipital ###################

# brain.add_label(occ_1_l, color='red')
# brain.add_label(occ_2_l, color='blue')
# brain.add_label(occ_3_l, color='yellow')
# brain.add_label(occ_4_l, color='green')
# brain.add_label(occ_5_l, color='purple')

# brain.add_label(occ_1_r, color='red')
# brain.add_label(occ_2_r, color='blue')
# brain.add_label(occ_3_r, color='yellow')
# brain.add_label(occ_4_r, color='green')
# brain.add_label(occ_5_r, color='purple')


################ affichage sous pericentral ###################

# brain.add_label(pericentral_1_l, color='blue')
# brain.add_label(pericentral_2_l, color='green')

# brain.add_label(pericentral_1_r, color='blue')
# brain.add_label(pericentral_2_r, color='green')

################ affichage sous pericentral ###################

# brain.add_label(ifg_1_l, color='cyan')

# brain.add_label(ifg_1_r, color='cyan')


################## affichage globale ##########################

TPJ_parietal_l = mne.Label(vertno_l[mask_TPJ_parietal_l], hemi='lh', name='test2')
occ_l = mne.Label(vertno_l[mask_occ_l], hemi='lh', name='test2')
temp_l =  mne.Label(vertno_l[mask_temp_l], hemi='lh', name='test')
pericentral_l = mne.Label(vertno_l[mask_pericentral_l], hemi='lh', name='test')

TPJ_parietal_r = mne.Label(vertno_r[mask_TPJ_parietal_r], hemi='rh', name='test2')
occ_r = mne.Label(vertno_r[mask_occ_r], hemi='rh', name='test2')
temp_r =  mne.Label(vertno_r[mask_temp_r], hemi='rh', name='test')
pericentral_r = mne.Label(vertno_r[mask_pericentral_r], hemi='rh', name='test')

brain.add_label(TPJ_parietal_l, color='lime')
brain.add_label(temp_l, color='yellow')
brain.add_label(occ_l, color='blue')
brain.add_label(pericentral_l, color='red')
brain.add_label(ifg_1_l, color='cyan')

brain.add_label(TPJ_parietal_r, color='lime')
brain.add_label(temp_r, color='yellow')
brain.add_label(occ_r, color='blue')
brain.add_label(pericentral_r, color='red')
brain.add_label(ifg_1_r, color='cyan')




